# Milestone 9 — Detection Evaluation

This notebook verifies the implementation of the evaluation infrastructure for cell detection (Sprint 10). It loads a baseline detection model, instantiates the validation dataloader, runs the `DetectionEvaluator` to compute coordinate-based precision, recall, and F1 score, prints the final metrics, and visualizes a prediction sample.

## 1. Environment Setup & Imports

In [ ]:
import sys
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import numpy as np

# Add project root to sys.path to allow importing src
project_root = str(Path("..").resolve())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.models.detection.baseline_detector import BaselineDetector
from src.datasets.dataloader import BiohubDataLoader
from src.losses.detection_loss import DetectionLoss
from src.postprocessing.peak_detector import PeakDetector
from src.metrics.detection_metrics import DetectionMetrics
from src.evaluation.evaluator import DetectionEvaluator
print("Imports loaded successfully.")

## 2. Locate Validation Dataset

In [ ]:
# Programmatically find the first dataset pair in data/raw/train
train_dir = Path("../data/raw/train")
zarr_paths = sorted(list(train_dir.glob("*.zarr")))
geff_paths = sorted(list(train_dir.glob("*.geff")))

test_zarr_path = None
test_geff_path = None

for z_path in zarr_paths:
    g_path = z_path.with_suffix(".geff")
    if g_path in geff_paths:
        test_zarr_path = z_path
        test_geff_path = g_path
        break

if not test_zarr_path or not test_geff_path:
    raise FileNotFoundError("Matching Zarr/GEFF pair not found in data/raw/train")

print(f"Using Zarr path: {test_zarr_path}")
print(f"Using GEFF path: {test_geff_path}")

## 3. Build Validation DataLoader

In [ ]:
bh_loader = BiohubDataLoader(
    test_zarr_path,
    test_geff_path,
    batch_size=1,
    shuffle=False,
)
val_loader = bh_loader.get_train_loader()
print(f"Validation DataLoader created with {len(val_loader)} batches.")

## 4. Setup Model & Mock Checkpoint Loading

In [ ]:
# Instantiate the model
model = BaselineDetector(in_channels=1, hidden_channels=16, out_channels=1)

# Create and save a mock checkpoint file to simulate loading a trained checkpoint
checkpoint_path = Path("temp_eval_checkpoint.pt")
checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": {},
    "scheduler_state_dict": {},
    "epoch": 10,
}
torch.save(checkpoint, checkpoint_path)
print(f"Saved mock checkpoint to {checkpoint_path}")

# Load checkpoint state dict back into model
loaded_checkpoint = torch.load(checkpoint_path, map_location="cpu")
model.load_state_dict(loaded_checkpoint["model_state_dict"])
print(f"Successfully loaded checkpoint from epoch {loaded_checkpoint['epoch']}")

# Clean up temporary checkpoint file
if checkpoint_path.exists():
    checkpoint_path.unlink()
    print("Temporary checkpoint file cleaned up.")

## 5. Initialize Evaluation Infrastructure

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    stream=sys.stdout
)

loss_fn = DetectionLoss()
peak_detector = PeakDetector(threshold=0.5)
metrics = DetectionMetrics(distance_threshold=15.0)

evaluator = DetectionEvaluator(
    model=model,
    val_loader=val_loader,
    peak_detector=peak_detector,
    metrics=metrics,
    loss_fn=loss_fn,
    device="cpu",
)
print("Evaluation components initialized successfully.")

## 6. Run Evaluator & Print Metrics Report

In [ ]:
# Run evaluation
summary = evaluator.evaluate()

print("\n" + "="*40)
print("        DETECTION EVALUATION REPORT")
print("="*40)
print(f"Mean Validation Loss: {summary['mean_loss']:.6f}")
print(f"Detection Precision:  {summary['precision']:.4f}")
print(f"Detection Recall:     {summary['recall']:.4f}")
print(f"Detection F1 Score:   {summary['f1_score']:.4f}")
print("="*40)

## 7. Visualize One Prediction Example

In [ ]:
# Get one validation batch
batch = next(iter(val_loader))
images = batch["image"]
targets = batch["target"]
positions = batch["positions"][0]

# Pre-process batch shapes for the model
if images.ndim == 4:
    images = images.unsqueeze(1)
if targets.ndim == 4:
    targets = targets.unsqueeze(1)

# Run inference
model.eval()
with torch.no_grad():
    predictions = model(images)

# Detect peaks
pred_heatmap = predictions[0]
pred_coords = peak_detector.detect(pred_heatmap) # (M, 3) tensor representing [z, y, x]

# GT positions coords [z, y, x] (column 1, 2, 3)
gt_coords = positions[:, 1:4] # (N, 3)

# Extract image spatial dimensions
_, _, Z, Y, X = images.shape
mid_z = Z // 2

# Extract slice corresponding to middle Z plane
img_slice = images[0, 0, mid_z].numpy()
target_slice = targets[0, 0, mid_z].numpy()
pred_slice = predictions[0, 0, mid_z].numpy()

# Plot slices
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_slice, cmap="gray")
axes[0].set_title(f"Input Image (Z Slice {mid_z})")
axes[0].axis("off")

# Plot targets and ground truth coordinates near slice
axes[1].imshow(target_slice, cmap="hot")
axes[1].set_title(f"GT Heatmap (Z Slice {mid_z})")
axes[1].axis("off")

# Plot predictions and overlay coordinates near this slice
axes[2].imshow(pred_slice, cmap="hot")
axes[2].set_title(f"Pred Heatmap (Z Slice {mid_z})")
axes[2].axis("off")

# Overlay coordinates falling in the vicinity of this Z plane (+/- 2 voxels)
near_gt = gt_coords[torch.abs(gt_coords[:, 0] - mid_z) <= 2]
near_pred = pred_coords[torch.abs(pred_coords[:, 0] - mid_z) <= 2] if len(pred_coords) > 0 else []

if len(near_gt) > 0:
    axes[1].scatter(near_gt[:, 2].numpy(), near_gt[:, 1].numpy(), color="cyan", marker="o", facecolors="none", s=80, label="GT Nodes")
    axes[1].legend()

if len(near_pred) > 0:
    axes[2].scatter(near_pred[:, 2].numpy(), near_pred[:, 1].numpy(), color="lime", marker="x", s=80, label="Pred Nodes")
    axes[2].legend()

plt.tight_layout()
plt.show()